In [10]:
from Influence_Modelling import Network, XsvNetwork, Sampling
import networkx as nx

def constant_threshold(G, n):
    return 0.5

class CelfHeuristic:
    def __init__(self, p: float = 0.1, q: float = 0.7, model: str = "ICC", thres_func = constant_threshold, runs=1000):
        self.p = p
        self.q = q
        self.model = model
        self.thres_func = thres_func
        self.__runs = runs
        
    def estimate_spread(self, net, seeds) -> float:
      total_activated = 0
      for _ in range(self.__runs):
            if self.model == "ICC":
                _, final, _ = net.run_ICC(seeds, self.p, 1)   # 1 run per iteration
            elif self.model == "DCC":
                _, final = net.run_DCC(seeds, p=self.p,
                                       dec_func=lambda x: x * self.q)
            elif self.model == "LTM":
                # Re‑initialise thresholds for each simulation (random per run)
                net.initialise_thresholds(self.thres_func)
                _, final = net.run_LTM(seeds)
            else:
                raise ValueError(f"Unknown model: {self.model}")

            # Count activated nodes (status == "used")
            activated = sum(1 for _, data in final.nodes(data=True)
                            if data.get("status") == "used")
            total_activated += activated
      return total_activated / self.__runs

    def __call__(self, G: nx.DiGraph, n: int = 2) -> list:
        marginal_gain = []
        nodes = [v for v in G.nodes()]
        net = Network.Network("")
        net.set_nx(G)

        for node in nodes:
            spread = self.estimate_spread(net, [node])
            marginal_gain.append((spread, node))

        marginal_gain.sort(reverse=True)
        Q = marginal_gain
        seeds = [Q[0]]
        spread = seeds[0][0]
        Q.pop(0)

        for _ in range(n - 1):
            check = False
            while not check:
                current = Q[0][1]
                seed_nodes = [s for (_, s) in seeds]
                new_spread = self.estimate_spread(net, seed_nodes + [current])
                Q[0] = (new_spread - spread, Q[0][1])
                Q.sort(reverse=True)
                check = Q[0][1] == current

            spread += Q[0][0]
            seeds.append(Q[0])
            print(len(seeds))
            Q = Q[1:]

        return [seed for (_, seed) in seeds]


In [6]:
X = XsvNetwork.XsvNetwork("physics")
Sample = Sampling.mhda_sample(X.get_nx(), 1000)
net = Network.Network("")
net.set_nx(Sample.copy())
print(net.get_nx())

Graph with 1000 nodes and 4219 edges


In [ ]:
celf = CelfHeuristic(runs=20)
celf(net.get_nx(), 100)

2
3
4
5
6
7
8
9
10


['99973',
 '99972',
 '99969',
 '99774',
 '99741',
 '99572',
 '99514',
 '99332',
 '9929',
 '99177']

In [44]:
cumulative_last = 0
for _ in range(100):
  _, _, x = net.run_ICC(['99973','99972','99969','99774','99741','99572','99514','99332','9929','99177'])
  cumulative_last += x[-1]
print(cumulative_last/100)

0.08304


In [63]:
def compute_shit(task_id):
  X = XsvNetwork.XsvNetwork("physics")
  Sample = Sampling.mhda_sample(X.get_nx(), 1000)
  net = Network.Network("")
  net.set_nx(Sample.copy())
  print(net.get_nx())
  celf = CelfHeuristic(runs=20)
  c = celf(net.get_nx(), 50)
  cumulative_last = 0
  for _ in range(100):
    _, _, x = net.run_ICC(c)
    cumulative_last += x[-1]
  return (cumulative_last/100)

compute_shit(1)

Graph with 1000 nodes and 4195 edges
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50


np.float64(0.20722)

In [56]:
import pickle
import concurrent.futures
from functools import partial
import time

# ------------------------------
# 2. Parallel execution and result saving
# ------------------------------
def run_parallel(func, num_runs=5, max_workers=5, **kwargs):
    """
    Runs `func` num_runs times in parallel using ProcessPoolExecutor.

    Parameters
    ----------
    func : callable
        Function with signature func(task_id, **kwargs)
    num_runs : int
        Number of parallel executions.
    max_workers : int, optional
        Number of processes (defaults to number of CPU cores).
    **kwargs :
        Additional arguments passed to `func` on every call.

    Returns
    -------
    list
        Results from all runs, in order of task_id (1..num_runs).
    """
    # Prepare the function with fixed arguments
    func_with_args = partial(func, **kwargs)

    results = [None] * num_runs
    with concurrent.futures.ProcessPoolExecutor(max_workers=max_workers) as executor:
        # Submit all tasks
        future_to_id = {
            executor.submit(func_with_args, task_id=i): i
            for i in range(1, num_runs + 1)
        }

        # Collect results as they complete
        for future in concurrent.futures.as_completed(future_to_id):
            task_id = future_to_id[future]
            try:
                results[task_id - 1] = future.result()
                print(f"Task {task_id} completed.")
            except Exception as e:
                print(f"Task {task_id} raised an exception: {e}")
                results[task_id - 1] = None

    return results


# ------------------------------
# 3. Save results to disk
# ------------------------------
def save_results(results, filename="results.pkl"):
    """Save results to a pickle file."""
    with open(filename, "wb") as f:
        pickle.dump(results, f)
    print(f"Results saved to {filename}")


# ------------------------------
# 4. Main execution
# ------------------------------
if __name__ == "__main__":
    # Example: run my_function 10 times with fixed arguments
    results = run_parallel(
        compute_shit,
        num_runs=6,
        max_workers=6   # optional, adjust as needed
    )

    # Save the list of results
    save_results(results, "parallel_results.pkl")

    # Optional: print summary
    print("All runs finished. Summary:")
    for i, res in enumerate(results, 1):
        print(f"  Run {i}: {res}")

Graph with 1000 nodes and 4836 edges
Graph with 1000 nodes and 4503 edges
Graph with 1000 nodes and 4160 edges
Graph with 1000 nodes and 4719 edges
Graph with 1000 nodes and 4357 edges
Graph with 1000 nodes and 4492 edges
2
3
4
5
6
7
8
9
10
Task 6 completed.
2
3
4
2
5
3
6
4
7
5
8
6
9
7
10
8
9
Task 2 completed.
10
Task 4 completed.
2
3
4
5
6
7
8
9
10
Task 3 completed.
2
3
4
5
6
7
8
9
10
Task 1 completed.
2
3
4
5
6
7
8
9
10
Task 5 completed.
Results saved to parallel_results.pkl
All runs finished. Summary:
  Run 1: 0.15247999999999992
  Run 2: 0.09104
  Run 3: 0.11748000000000003
  Run 4: 0.15142
  Run 5: 0.1541899999999999
  Run 6: 0.13346


In [62]:
import numpy as np
import pandas as pd
x = pd.read_pickle("parallel_results.pkl")
np.sum(x)/6

np.float64(0.13334499999999996)